In [ ]:
# Step 1: Installing essential AI libraries for my LLM engineering journey.
# 'transformers' is crucial for working with large language models (LLMs) and vision models from Hugging Face.
# 'diffusers' is specifically for generative models, which I'll use for image generation.
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2

In [ ]:
# Step 2: Checking for GPU availability.
# Generative AI, especially with LLMs, demands significant computational power, and GPUs are much faster than CPUs.
# I'm using '!nvidia-smi' to verify if a 'Tesla T4' GPU (commonly available in Colab) is connected, which is vital for efficient model inference.
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU, which might slow down my generative AI tasks.')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4, perfect for getting started!")
  else:
    print("NOT CONNECTED TO A T4 - performance might be impacted for larger models.")

In [ ]:
# Step 3: Logging into Hugging Face.
# Many advanced models, like SDXL or Flux, require accepting terms and using an API token.
# This code helps me log in automatically using my stored Hugging Face token, which is essential for accessing these models.
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Step 4: Generating an image using 'SDXL Turbo' – my first generative AI output!
# Turbo models are optimized for speed, producing high-quality images in very few inference steps (e.g., just 4 steps here).
# I'm loading the model, moving it to the GPU (cuda) for speed, and then providing a text prompt to guide the image generation.
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Maintenance: Restarting the kernel to clear system memory (RAM).
# This is a common practice in LLM engineering to ensure a clean slate and free up resources before loading a new, potentially larger, model.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 5: Generating an image with the full SDXL Base model.
# This model is more powerful than the Turbo version, offering higher quality, but it's also slower (requiring around 30 inference steps).
# It's a good trade-off to understand the balance between speed and quality in generative models.
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=30).images[0]

display(image)

In [ ]:
# Maintenance: Shutting down the kernel again to free up VRAM.
# This is crucial for managing GPU memory, especially when I'm about to load another model configuration that might be VRAM-intensive.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 6: Exploring the Base + Refiner strategy for enhanced image generation.
# This advanced technique uses two models in tandem: the 'Base' model establishes the overall structure, and the 'Refiner' model then adds intricate details.
# I'm passing latent data (a compressed representation) from the base model to the refiner to achieve higher fidelity.
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

In [ ]:
# Maintenance: Clearing memory after running the dual-model setup.
# This ensures that all VRAM is freed up after complex multi-model operations, preparing the environment for subsequent tasks.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 7: Installing the 'datasets' library.
# This library is incredibly useful for downloading and managing various datasets from Hugging Face, whether for training LLMs, working with audio samples, or text data.
!pip install --upgrade datasets==3.6.0

In [ ]:
# Step 8: A Text-to-Speech (TTS) demonstration – bringing LLM outputs to life with audio!
# I'm using a 'pipeline' from Hugging Face to convert text into speech with the Microsoft SpeechT5 model.
# The cool part is using a 'speaker embedding' to customize the voice's characteristics, making the output more dynamic.
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
# Maintenance: Clearing memory to prepare for the high-end Flux model.
# This step is critical because Flux is a very demanding model, and I need to ensure maximum VRAM is available for its operation.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Step 9: Verifying high-end hardware for advanced models.
# For state-of-the-art models like Flux, a powerful GPU like the NVIDIA A100 is often a prerequisite for optimal performance.
# I'm checking if I have access to such hardware, which is key for pushing the boundaries of generative AI.
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU, this will limit my ability to run advanced models.')
else:
  print(gpu_info)
  if gpu_info.find('A100') >= 0:
    print("Success - Connected to an NVIDIA A100, ready for some serious generative AI!")
  else:
    print("NOT CONNECTED TO AN A100 - performance for high-end models like Flux might be suboptimal.")

In [ ]:
# Step 10: Generating an image using FLUX.1 [schnell] – trying out a cutting-edge model!
# Flux is a state-of-the-art generative model, and 'schnell' indicates the fast version.
# I'm also tracking the time taken for this process using the 'datetime' library, which is important for understanding computational costs in LLM engineering.
import torch
from diffusers import FluxPipeline
from IPython.display import display
from datetime import datetime

start = datetime.now()

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
generator = torch.Generator(device="cuda").manual_seed(0)
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator
).images[0]

display(image)

stop = datetime.now()

In [ ]:
# Step 11: Calculating the estimated cost of this model run.
# Cloud GPUs, especially high-end ones like A100s, have hourly rates. This calculation converts the processing time into an estimated dollar amount.
# This is a practical aspect of LLM engineering, as efficient resource management directly impacts project costs.
seconds = (stop-start).total_seconds()
units_per_hour = 5.37
estimated_units = (5.37 / 3600) * seconds
estimated_cost = estimated_units * (9.99/100) # Assuming some rate, this helps me understand cost implications.
print(f"This model run took {seconds:.1f} seconds and had an estimated cost of ${estimated_cost:.3f}")